In [ ]:
file_path = "Files/raw/Dataverse/05/2026/msemr_referralrequest.parquet"

df = spark.read.parquet(file_path)

column_names = df.columns

print(column_names)

In [ ]:
p_file1_alias = "msemr_referralrequest"
p_file1_path = "Files/raw/Dataverse/05/2026/msemr_referralrequest.parquet"
p_file1_format = "parquet"

p_file2_alias = "mmhci_incomingreferral"
p_file2_path = "Files/raw/Dataverse/05/2026/mmhci_incomingreferral.parquet"
p_file2_format = "parquet"

p_similarity_threshold = "0.75"
p_save_results = "true"
p_output_table_prefix = "md_eferralRequest_incomingreferral_join_check"

In [ ]:
from difflib import SequenceMatcher
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
import json
import re

# ============================================================
# 1. Convert parameters
# ============================================================

similarity_threshold = float(p_similarity_threshold)
save_results = str(p_save_results).lower() == "true"
output_table_prefix = p_output_table_prefix.strip()

# ============================================================
# 2. Validate parameters
# ============================================================

if not p_file1_path or p_file1_path.strip() == "":
    raise ValueError("p_file1_path is required.")

if not p_file2_path or p_file2_path.strip() == "":
    raise ValueError("p_file2_path is required.")

files_config = [
    {
        "file_alias": p_file1_alias,
        "file_path": p_file1_path,
        "file_format": p_file1_format
    },
    {
        "file_alias": p_file2_alias,
        "file_path": p_file2_path,
        "file_format": p_file2_format
    }
]

print("Files received for join key comparison:")
for file_config in files_config:
    print(file_config)

# ============================================================
# 3. Helper functions
# ============================================================

def normalize_column_name(column_name: str) -> str:
    """
    Normalize column names to improve comparison.
    Example:
    Client Key -> client_key
    client-key -> client_key
    Client.ID  -> client_id
    """

    if column_name is None:
        return ""

    normalized = column_name.strip().lower()
    normalized = re.sub(r"[\s\-.]+", "_", normalized)
    normalized = re.sub(r"[^a-z0-9_]", "", normalized)
    normalized = re.sub(r"_+", "_", normalized)
    normalized = normalized.strip("_")

    return normalized


def similarity_score(column_1: str, column_2: str) -> float:
    """
    Returns fuzzy similarity score between two normalized column names.
    Score range: 0 to 1.
    """

    return SequenceMatcher(
        None,
        normalize_column_name(column_1),
        normalize_column_name(column_2)
    ).ratio()


def has_join_keyword(column_name: str) -> bool:
    """
    Checks whether a column name looks like a possible join key.
    """

    join_keywords = [
        "id",
        "key",
        "code",
        "number",
        "uuid",
        "session",
        "client",
        "visitor",
        "interaction",
        "account",
        "customer",
        "user",
        "member",
        "person",
        "record"
    ]

    normalized = normalize_column_name(column_name)

    return any(keyword in normalized for keyword in join_keywords)


def read_file_schema(file_path: str, file_format: str):
    """
    Reads schema from supported file formats:
    parquet, csv, json, delta.
    """

    file_format = file_format.lower().strip()

    if file_format == "parquet":
        df = spark.read.parquet(file_path)

    elif file_format == "csv":
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

    elif file_format == "json":
        df = spark.read.json(file_path)

    elif file_format == "delta":
        df = spark.read.format("delta").load(file_path)

    else:
        raise ValueError(f"Unsupported file format: {file_format}")

    columns_with_types = []

    for index, field in enumerate(df.schema.fields, start=1):
        columns_with_types.append(
            {
                "column_position": index,
                "column_name": field.name,
                "normalized_column_name": normalize_column_name(field.name),
                "data_type": field.dataType.simpleString()
            }
        )

    return columns_with_types


# ============================================================
# 4. Read schemas for both files
# ============================================================

schema_dict = {}

for file_config in files_config:
    file_alias = file_config["file_alias"]
    file_path = file_config["file_path"]
    file_format = file_config["file_format"]

    print(f"Reading schema for {file_alias}: {file_path}")

    try:
        schema_dict[file_alias] = read_file_schema(file_path, file_format)
        print(f"Successfully read schema for {file_alias}")

    except Exception as ex:
        raise Exception(
            f"Failed to read schema for alias={file_alias}, path={file_path}. Error: {str(ex)}"
        )

file1_alias = p_file1_alias
file2_alias = p_file2_alias

file1_columns = schema_dict[file1_alias]
file2_columns = schema_dict[file2_alias]

# ============================================================
# 5. Display discovered columns
# ============================================================

schema_rows = []

for file_alias, columns in schema_dict.items():
    file_path = next(
        item["file_path"] for item in files_config
        if item["file_alias"] == file_alias
    )

    for column in columns:
        schema_rows.append(
            Row(
                file_alias=file_alias,
                file_path=file_path,
                column_position=column["column_position"],
                column_name=column["column_name"],
                normalized_column_name=column["normalized_column_name"],
                data_type=column["data_type"]
            )
        )

schema_df = spark.createDataFrame(schema_rows)

print("Discovered columns from both files:")
display(schema_df.orderBy("file_alias", "column_position"))

# ============================================================
# 6. Find exact matching columns
# ============================================================

file1_column_map = {
    column["normalized_column_name"]: column
    for column in file1_columns
}

file2_column_map = {
    column["normalized_column_name"]: column
    for column in file2_columns
}

exact_common_columns = set(file1_column_map.keys()).intersection(
    set(file2_column_map.keys())
)

exact_match_rows = []

for normalized_column in sorted(exact_common_columns):
    col1 = file1_column_map[normalized_column]
    col2 = file2_column_map[normalized_column]

    exact_match_rows.append(
        Row(
            left_file=file1_alias,
            left_column=col1["column_name"],
            left_data_type=col1["data_type"],
            right_file=file2_alias,
            right_column=col2["column_name"],
            right_data_type=col2["data_type"],
            normalized_column_name=normalized_column,
            data_type_match=(
                col1["data_type"].lower() == col2["data_type"].lower()
            ),
            suggested_join_condition=(
                f"{file1_alias}.{col1['column_name']} = "
                f"{file2_alias}.{col2['column_name']}"
            )
        )
    )

if exact_match_rows:
    exact_match_df = spark.createDataFrame(exact_match_rows)
    print("Exact matching columns:")
    display(exact_match_df)
else:
    exact_match_df = None
    print("No exact matching column names found.")

# ============================================================
# 7. Find similar columns using fuzzy matching
# ============================================================

similar_column_rows = []

for col1 in file1_columns:
    for col2 in file2_columns:
        score = similarity_score(col1["column_name"], col2["column_name"])

        if score >= similarity_threshold:
            similar_column_rows.append(
                Row(
                    left_file=file1_alias,
                    left_column=col1["column_name"],
                    left_normalized_column=col1["normalized_column_name"],
                    left_data_type=col1["data_type"],
                    right_file=file2_alias,
                    right_column=col2["column_name"],
                    right_normalized_column=col2["normalized_column_name"],
                    right_data_type=col2["data_type"],
                    similarity_score=round(score, 3),
                    data_type_match=(
                        col1["data_type"].lower() == col2["data_type"].lower()
                    ),
                    suggested_join_condition=(
                        f"{file1_alias}.{col1['column_name']} = "
                        f"{file2_alias}.{col2['column_name']}"
                    )
                )
            )

if similar_column_rows:
    similar_columns_df = spark.createDataFrame(similar_column_rows)
    print("Similar columns:")
    display(
        similar_columns_df.orderBy(
            "similarity_score",
            ascending=False
        )
    )
else:
    similar_columns_df = None
    print("No similar columns found based on threshold.")

# ============================================================
# 8. Suggest likely join keys
# ============================================================

suggested_join_rows = []

for col1 in file1_columns:
    for col2 in file2_columns:
        score = similarity_score(col1["column_name"], col2["column_name"])

        keyword_match = (
            has_join_keyword(col1["column_name"])
            or has_join_keyword(col2["column_name"])
        )

        data_type_match = (
            col1["data_type"].lower() == col2["data_type"].lower()
        )

        exact_match = (
            col1["normalized_column_name"] == col2["normalized_column_name"]
        )

        # Strong suggestion rules:
        # 1. Exact column name match and looks like a join key
        # OR
        # 2. Similar enough and looks like a join key
        if (exact_match and keyword_match) or (
            score >= similarity_threshold and keyword_match
        ):
            confidence = "high" if exact_match and data_type_match else "medium"

            if not data_type_match:
                confidence = "low"

            suggested_join_rows.append(
                Row(
                    left_file=file1_alias,
                    left_column=col1["column_name"],
                    left_data_type=col1["data_type"],
                    right_file=file2_alias,
                    right_column=col2["column_name"],
                    right_data_type=col2["data_type"],
                    similarity_score=round(score, 3),
                    data_type_match=data_type_match,
                    confidence=confidence,
                    suggested_join_condition=(
                        f"{file1_alias}.{col1['column_name']} = "
                        f"{file2_alias}.{col2['column_name']}"
                    )
                )
            )

if suggested_join_rows:
    suggested_join_df = spark.createDataFrame(suggested_join_rows)
    print("Suggested join keys:")
    display(
        suggested_join_df.orderBy(
            "confidence",
            "similarity_score",
            ascending=False
        )
    )
else:
    suggested_join_df = None
    print("No suggested join keys found.")

# ============================================================
# 9. Save output tables if required
# ============================================================

if save_results:
    schema_table_name = f"{output_table_prefix}_discovered_schema"
    exact_match_table_name = f"{output_table_prefix}_exact_matches"
    similar_table_name = f"{output_table_prefix}_similar_matches"
    suggested_join_table_name = f"{output_table_prefix}_suggested_joins"

    schema_df.withColumn("captured_at", current_timestamp()) \
        .write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(schema_table_name)

    if exact_match_df is not None:
        exact_match_df.withColumn("captured_at", current_timestamp()) \
            .write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(exact_match_table_name)

    if similar_columns_df is not None:
        similar_columns_df.withColumn("captured_at", current_timestamp()) \
            .write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(similar_table_name)

    if suggested_join_df is not None:
        suggested_join_df.withColumn("captured_at", current_timestamp()) \
            .write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(suggested_join_table_name)

    print("Results saved to Lakehouse tables:")
    print(schema_table_name)
    print(exact_match_table_name)
    print(similar_table_name)
    print(suggested_join_table_name)

# ============================================================
# 10. Return summary to pipeline
# ============================================================

summary = {
    "status": "success",
    "left_file": file1_alias,
    "right_file": file2_alias,
    "exact_match_count": len(exact_match_rows),
    "similar_match_count": len(similar_column_rows),
    "suggested_join_count": len(suggested_join_rows),
    "similarity_threshold": similarity_threshold,
    "results_saved": save_results
}

notebookutils.notebook.exit(json.dumps(summary))